# Automatically find all person names in your reference lists and paragraphs and save them with their source file and volume information.


#### 1. Imports

In [1]:
import pandas as pd
import spacy

# run this once in terminal if you haven't already:
# python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm")

I decided for spacy because: 

Using this: https://www.geeksforgeeks.org/python/python-named-entity-recognition-ner-using-spacy/

#### 2. Loading the H2 Corpus

In [2]:
df = pd.read_csv("/Users/sophiehamann/master-thesis-code/data/output_files/07_h2_normalized.csv")
print(df.shape)
print(df["region_type"].value_counts())

(12620, 5)
region_type
paragraph         12320
reference_list      300
Name: count, dtype: int64


#### 3. Assigning volume groups

In [3]:
import re

def get_issue_number(filepath):
    match = re.search(r"heresies_(\d+)_combined", filepath)
    return int(match.group(1)) if match else None

def get_volume(issue_nr):
    if issue_nr <= 4:
        return "Vol1_late1970s"
    elif issue_nr <= 8:
        return "Vol2_early1980s"
    elif issue_nr <= 12:
        return "Vol3_late1980s"
    elif issue_nr <= 16:
        return "Vol4_early1990s"
    elif issue_nr <= 20:
        return "Vol5"
    elif issue_nr <= 24:
        return "Vol6"
    else:
        return "Vol7"

df["issue"]  = df["source_file"].apply(get_issue_number)
df["volume"] = df["issue"].apply(get_volume)

# only work with issues 1-14 for now
df = df[df["issue"] <= 14]

print(df["volume"].value_counts())

volume
Vol2_early1980s    4320
Vol1_late1970s     3732
Vol3_late1980s     3611
Vol4_early1990s     957
Name: count, dtype: int64


#### 4. Extracting person names
Using this: https://spacy.io/usage/linguistic-features#named-entities

In [4]:
def extract_persons(text):
    if pd.isna(text):
        return []
    
    doc = nlp(text)
    
    # only keep PERSON entities
    persons = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]
    
    return persons

df["persons"] = df["text"].apply(extract_persons)
print("done")
print(df["persons"].iloc[0])

done
['Editorial Collective', 'Lula Mae Blocton', 'Yvonne Flowers', 'Valerie Harris', 'Zarina Hashmi']


#### 5. Checking what has been found
Using this: https://www.geeksforgeeks.org/python/python-counter-objects-elements/

In [5]:
# flatten all person names into one list
all_persons = [person for persons in df["persons"] for person in persons]

# count how often each name appears
from collections import Counter
person_counts = Counter(all_persons)

print(f"Total person mentions found: {len(all_persons)}")
print(f"Unique names found: {len(person_counts)}")
print("\nTop 30 most mentioned people:")
for name, count in person_counts.most_common(30):
    print(f"  {name}: {count}")

Total person mentions found: 2026
Unique names found: 1472

Top 30 most mentioned people:
  Martine: 26
  Vicki: 20
  Donna: 18
  Mia: 17
  Jean: 15
  Patsy Beckert: 13
  Su Friedrich: 13
  Ruth: 12
  Joan Braderman: 11
  Alice: 11
  Renée: 11
  Lacey: 11
  Lee: 11
  Edie: 10
  Martha: 10
  Michelle: 9
  Chicanas: 8
  Ann: 8
  MA: 8
  Myrna Zimmerman: 7
  Lyn Blumenthal: 7
  Cynthia Carr: 7
  Editorial Collective: 6
  Sue Heinemann: 6
  Ruthy: 6
  Bessie Smith: 6
  Harmony Hammond: 6
  Katie: 6
  Jacki: 6
  Sharon: 5


In [11]:
def clean_persons(persons):
    cleaned = []
    for name in persons:
        # remove possessives
        name = name.replace("'s", "").strip()
        
        # skip single word names — too ambiguous
        if len(name.split()) < 2:
            continue
        
        # skip names with numbers or special characters
        if any(char.isdigit() for char in name):
            continue
            
        # skip short names under 5 characters total
       # if len(name) < 5:
        #    continue
            
        cleaned.append(name)
    return cleaned

df["persons"] = df["persons"].apply(clean_persons)

#### 6. Create one row per person mention with metadata

In [14]:
rows = []

for _, row in df.iterrows():
    for person in row["persons"]:
        rows.append({
            "source_file": row["source_file"],
            "issue":       row["issue"],
            "volume":      row["volume"],
            "region_type": row["region_type"],
            "person":      person
        })

df_persons = pd.DataFrame(rows)
print(df_persons.shape)
print(df_persons.head(30))

(1251, 5)
                                          source_file  issue           volume  \
0   /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_early1980s   
1   /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_early1980s   
2   /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_early1980s   
3   /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_early1980s   
4   /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_early1980s   
5   /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_early1980s   
6   /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_early1980s   
7   /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_early1980s   
8   /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_early1980s   
9   /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_early1980s   
10  /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_early1980s   
11  /Users/sophieh

In [17]:
# list of known false positives to remove
false_positives = {
    "Editorial Collective", "De Colores", "Sacred Song Sung",
    "Kitchen Dreams", "Kitchen Artists", "Kitchens", "Iris Films"
}

name_mapping = {
    "Ida b. Wells":     "Ida B. Wells-Barnett",
    "Ida B. Wells":     "Ida B. Wells-Barnett",
    "Lucy R. Lippard":  "Lucy Lippard",
}

def clean_name(name):
    # remove false positives
    if name in false_positives:
        return None
    # standardize duplicates
    if name in name_mapping:
        return name_mapping[name]
    return name

df_persons["person"] = df_persons["person"].apply(clean_name)
df_persons = df_persons.dropna(subset=["person"])

print(f"Cleaned person mentions: {len(df_persons)}")
print(df_persons["person"].value_counts().head(30))

Cleaned person mentions: 1238
person
Patsy Beckert          13
Su Friedrich           13
Joan Braderman         11
Lucy Lippard            8
Harmony Hammond         8
Cynthia Carr            7
Lyn Blumenthal          7
Myrna Zimmerman         7
Bessie Smith            7
Sue Heinemann           6
Mary Beth Edelson       6
Janet Froelich          5
Adrienne Rich           4
Louis Sullivan          4
Leslie Labowitz         4
Suzanne Lacy            4
Virginia Woolf          4
Howardena Pindell       3
Linda Nochlin           3
Ruth Crawford           3
Batya Weinbaum          3
Martha Wilson           3
Beth Anderson           3
Elizabeth Hess          3
Martha Rosler           3
Barbara Ehrenreich      3
Marty Pottenger         3
Mimi Smith              3
Claire Pajaczkowska     3
Toni Robertson          3
Name: count, dtype: int64


#### 7. Saving

In [18]:
df_persons.to_csv("/Users/sophiehamann/master-thesis-code/data/output_files/09_h2_persons.csv", index=False)